[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module1/2_AllTheModels_Regression.ipynb)

# Module 1.2 - All the Models (Regression)

**OPIM 5509: Introduction to Deep Learning - University of Connecticut**

Same data, now with models on top. We predict `MedHouseVal` - a continuous number - so this is a **regression** problem.

Watch what happens as we go: we set up the data once, and then every model is three lines. Instantiate, fit, predict. Linear regression, a decision tree, a random forest, gradient boosting - the code barely changes.

That is the point of this notebook. In Module 2 the model object becomes a **neural network**, and the surrounding code stays exactly the same.

🔷 **The nugget:** read -> split -> scale on train -> fit -> evaluate. Memorize this skeleton. Everything else this semester is a substitution into it.

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
Video 5 - Regression Part 1 - the skeleton: split, scale, and leakage

- OPEN with the promise of this notebook: "by the end you will have fit five models and the code for the
  fifth will look identical to the code for the first." That predictability IS the skill.
- Rebuild the data quickly - do not re-teach EDA, just load and apply the cleaning we justified in notebook 1.
- X and y. Say it plainly: y is what we predict, X is everything else. Show the shapes and make them
  say out loud why X lost a column.
- train_test_split: 80/20, shuffle=True, random_state. SPEND TIME ON random_state. Without it your numbers
  change every run and you cannot tell a real improvement from noise. It is not about "the right answer,"
  it is about reproducibility.
- DATA LEAKAGE is the centerpiece of this video. Draw it out:
    * fit_transform on TRAIN - the scaler learns min and max from training data only
    * transform on TEST - test data gets the training scaler applied to it
  Then say the wrong version out loud: "if I scale before splitting, the test set's maximum leaks into
  the training scaler, and my test score is now a lie." This is THE most common mistake in student
  final projects. Say that sentence.
- Show the tmp.describe() check - all train mins are 0, all maxes are 1. And point out that TEST can go
  slightly outside 0 to 1. That is correct and expected, not a bug. Students always ask.
- Note we do NOT scale y for regression. Mention that in Module 2 we sometimes will, and why.
-->

## 1. Setup and data

We reload California Housing and apply the cleaning we justified back in notebook 1 - drop the small-denominator artifacts, keep the capped rows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# splitting and scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# the models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

# how we score them
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.datasets import fetch_california_housing

df = fetch_california_housing(as_frame=True).frame

# the cleaning we justified in notebook 1
df = df[(df["AveOccup"] <= 10) & (df["AveRooms"] <= 20)].copy()

print("Shape after cleaning:", df.shape)
df.head(3)

## 2. Split X from y

`y` is what we are trying to predict. `X` is everything else.

In [ ]:
y = df["MedHouseVal"]                     # the target - one column
X = df.drop(columns="MedHouseVal")       # the features - everything else

print("y shape:", y.shape)
print("X shape:", X.shape, "  <- one fewer column than df, because the target moved out")
print()
print("Feature names:", list(X.columns))

## 3. Train/test split

An 80/20 split, shuffled, with a fixed `random_state`.

**Remember:** `random_state` is not superstition. Without it, every run gives you different numbers and you can never tell whether a change actually helped or you just got a luckier shuffle.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20% held out
    shuffle=True,        # shuffle before splitting
    random_state=42,     # reproducible - same split every time
)

print("X_train:", X_train.shape, "   y_train:", y_train.shape)
print("X_test :", X_test.shape, "   y_test :", y_test.shape)

In [ ]:
# Keep the column names for later, then convert to numpy arrays.
feature_names = list(X_train.columns)

X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

print("Now they are numpy arrays:", type(X_train).__name__, X_train.shape)

## 4. Scaling - and the leakage trap

`MinMaxScaler` squeezes every feature into the range 0 to 1. Tree models do not need it; linear models, k-nearest neighbours, and **every neural network you build this semester** absolutely do.

The order of operations is not negotiable:

- **`fit_transform(X_train)`** - the scaler *learns* the min and max **from the training data only**, then applies them
- **`transform(X_test)`** - the test data gets the *training* scaler applied to it, and teaches the scaler nothing

**Caution:** if you scale *before* splitting, the test set's minimum and maximum leak into the scaler, your model has secretly seen the test data, and your test score becomes a lie. This is the single most common mistake in student final projects. Split first. Always.

In [ ]:
scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)   # LEARN from train, then apply
X_test = scaler.transform(X_test)         # only APPLY to test

# Check your work: every training column should now run from 0 to 1
pd.DataFrame(X_train, columns=feature_names).describe().T[["min", "max"]].round(3)

In [ ]:
# And the test set? Slightly outside 0 to 1 - which is CORRECT, not a bug.
# A test value more extreme than anything in training lands outside the range. That is the honest answer.
pd.DataFrame(X_test, columns=feature_names).describe().T[["min", "max"]].round(3)

Notice we did **not** scale `y`. For regression with these models it is unnecessary. In Module 2 you will sometimes scale the target for a neural network, and we will talk about why then.

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
Video 6 - Regression Part 2 - fitting five models and reading the results

- OPEN by pointing at the code: instantiate, fit, predict. Three lines. Now do it five times.
- Fit them one at a time on screen so they see the pattern, THEN show the loop that does all five.
  The loop is the payoff - "this is what it looks like when you stop copy-pasting."
- Metrics, and what each one is FOR:
    * MAE - average miss, in the target's own units. $100,000s here. Most interpretable. Lead with it.
    * RMSE - same units but punishes big misses harder. Use when large errors are expensive.
    * R2 - proportion of variance explained. Unitless, so it travels between problems.
  Say the MAE out loud in dollars: "0.31 means we are off by about thirty-one thousand dollars on a typical
  block group." Making the number CONCRETE is the whole job.
- TRAIN vs TEST is the second big idea. Show the decision tree: near-perfect on train, mediocre on test.
  That gap IS overfitting, made visible. Then random forest - same family, much smaller gap. Explain that
  averaging many trees is what closes it.
- Warn about reading R2 on train. Students report train R2 in projects every single semester. Do not.
- The predicted-vs-actual plot: a good model hugs the 45 degree line. Point at the horizontal stripe of
  points at 5.0 - THAT IS THE CENSORING FROM NOTEBOOK 1, showing up in the residuals. Beautiful payoff.
  No model can beat it, and now they can see why.
- Feature importance: MedInc dominates, then the geography columns. Ties straight back to the EDA map.
- CLOSE with the bridge to Module 2: "next module the model object becomes Sequential([Dense(...)]).
  Everything above and below that line stays exactly the same." Show them the skeleton one more time.
-->

## 5. Fit the models

Instantiate, fit, predict. Three lines each. Here is the first one in full.

In [ ]:
# Linear regression - our baseline. Always have a baseline.
LR = LinearRegression()               # 1. instantiate
LR = LR.fit(X_train, y_train)         # 2. fit on the TRAINING data
train_preds = LR.predict(X_train)     # 3. predict
test_preds = LR.predict(X_test)

print("MAE on test:", round(mean_absolute_error(y_test, test_preds), 4))
print("R2  on test:", round(r2_score(y_test, test_preds), 4))

That is the entire pattern. Now watch it repeat - the only thing that changes is the object on line 1.

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    """Fit a model and return its train/test scores as a dict."""
    model.fit(X_tr, y_tr)
    tr_pred = model.predict(X_tr)
    te_pred = model.predict(X_te)
    return {
        "Model": name,
        "Train MAE": mean_absolute_error(y_tr, tr_pred),
        "Test MAE": mean_absolute_error(y_te, te_pred),
        "Test RMSE": np.sqrt(mean_squared_error(y_te, te_pred)),
        "Train R2": r2_score(y_tr, tr_pred),
        "Test R2": r2_score(y_te, te_pred),
    }


models = {
    "Linear Regression":  LinearRegression(),
    "Decision Tree":      DecisionTreeRegressor(random_state=42),
    "Random Forest":      RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting":  GradientBoostingRegressor(random_state=42),
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=10),
}

rows = []
for name, model in models.items():
    print("Fitting", name, "...")
    rows.append(evaluate(name, model, X_train, y_train, X_test, y_test))

results = pd.DataFrame(rows).set_index("Model").round(4)
print("\nDone.")
results

## 6. Reading that table

Three metrics, three jobs:

| Metric | What it tells you |
| --- | --- |
| **MAE** | The average miss, in the target's own units. Most interpretable - lead with this one. |
| **RMSE** | Same units, but squares the errors first, so it punishes large misses harder. |
| **R2** | The share of variance the model explains. Unitless, so it compares across problems. |

Say the MAE out loud in real money. The target is in hundreds of thousands of dollars, so a test MAE of `0.31` means **we are off by roughly \$31,000 on a typical block group**. That sentence is worth more than the number.

In [ ]:
# Put the errors in dollars - this is the number a stakeholder actually understands.
for name, row in results.iterrows():
    print(f"{name:<22} typical miss: ${row['Test MAE'] * 100_000:,.0f}")

## 7. Train versus test: seeing overfitting

Now look at the `Train R2` column next to `Test R2`.

The **decision tree** scores near-perfect on training data and much worse on test. It memorized. That gap *is* overfitting, and here it is as a number rather than a definition.

The **random forest** is the same kind of model - but it averages a hundred trees, each grown on a different bootstrap sample, and the gap collapses.

**Caution:** never report training R2 as your result. It measures how well the model memorized the answers it was already shown.

In [ ]:
gap = (results["Train R2"] - results["Test R2"]).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
gap.plot(kind="barh", color="#C0202C")
plt.xlabel("Train R2 minus Test R2  (bigger = more overfitting)")
plt.title("The overfitting gap")
plt.tight_layout()
plt.show()

gap.round(4)

## 8. Predicted versus actual

Numbers tell you *which* model wins. This plot tells you *how* it is wrong - and that is usually the more useful information. A good model hugs the 45-degree line.

In [ ]:
best_name = results["Test R2"].idxmax()
best_model = models[best_name]
best_preds = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, (name, model) in zip(axes, [("Linear Regression", models["Linear Regression"]),
                                    (best_name, best_model)]):
    preds = model.predict(X_test)
    ax.scatter(y_test, preds, alpha=0.15, s=8, color="#C0202C")
    lims = [0, 5.2]
    ax.plot(lims, lims, "k--", linewidth=1.2)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("Actual MedHouseVal")
    ax.set_ylabel("Predicted MedHouseVal")
    ax.set_title(f"{name}  (test R2 = {r2_score(y_test, preds):.3f})")

plt.suptitle("Predicted vs. actual - a good model hugs the dashed line")
plt.tight_layout()
plt.show()

Look along the right edge of both plots: a vertical stripe of points at `actual = 5.0` with predictions scattered below it.

That is the **censoring from notebook 1**, reappearing in the residuals. Those block groups are all recorded at the \$500,001 ceiling, and no model can predict past it. You found this problem during EDA, and here is exactly what it costs you.

🔷 **The nugget:** the flaws you find in EDA do not go away when you start modeling. They show up in the residuals. That is why we look first.

## 9. Which features actually mattered?

In [ ]:
rf = models["Random Forest"]

importance = (pd.Series(rf.feature_importances_, index=feature_names)
                .sort_values(ascending=True))

plt.figure(figsize=(8, 4.5))
importance.plot(kind="barh", color="#C0202C")
plt.xlabel("Random forest feature importance")
plt.title("What the forest leaned on")
plt.tight_layout()
plt.show()

importance.sort_values(ascending=False).round(4)

`MedInc` dominates - exactly what the correlation heatmap predicted in notebook 1. `Latitude` and `Longitude` come next, which is the map you drew, rediscovered by the model on its own.

**Caution:** built-in tree importance is biased toward high-cardinality features. `permutation_importance` is the more trustworthy tool - it shuffles one column at a time and measures how much the score drops.

In [ ]:
from sklearn.inspection import permutation_importance

# Shuffle each column and see how much the test score degrades. Slower, but more honest.
perm = permutation_importance(rf, X_test, y_test, n_repeats=5,
                              random_state=42, n_jobs=-1)

(pd.Series(perm.importances_mean, index=feature_names)
   .sort_values(ascending=False)
   .round(4))

## 10. The bridge to Module 2

Scroll back through this notebook and look at what actually changed between models:

```python
LR = LinearRegression()
DT = DecisionTreeRegressor(random_state=42)
RF = RandomForestRegressor(n_estimators=100, random_state=42)
```

One line. Everything around it - the split, the scaler, the fit, the predictions, the metrics, the predicted-vs-actual plot - stayed identical.

Next module, that one line becomes:

```python
model = Sequential([Dense(64, activation="relu"), Dense(1)])
```

and the rest of this notebook still works. That is why we spent Module 1 here.

## What you should have after this notebook

- The skeleton memorized: **read -> split -> scale on train -> fit -> evaluate**
- You can explain data leakage and say precisely why `fit_transform` belongs only on train
- You can pick between MAE, RMSE, and R2 and justify the choice
- You can spot overfitting from a train/test gap without being told it is there
- You can read a predicted-vs-actual plot and connect its patterns back to your EDA
- You know the difference between built-in and permutation feature importance

**On your own:** re-run the whole notebook with `random_state=7` in the split. How much do the test scores move? That wobble is your noise floor - any "improvement" smaller than it is not real.

---

**Next:** `3_AllTheModels_Classification.ipynb` - the same skeleton, a different question.